# Evaluation

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/07_evaluation.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️

## Setup

### Housekeeping (no interaction required)
You only need to execute the cells (press <img src="https://api.iconify.design/icon-park-solid/play.svg" alt="▶️" width=12> to the left of the cells).

In [ ]:
%pip install scikit-learn seaborn

❗ Please restart the kernel/runtime after installing the packages to ensure that all changes take effect.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    cohen_kappa_score,
)

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBSummerSchool26/data') if IN_COLAB else Path('../data')

### Setup (interaction required)

In [ ]:
### ⬇️⬇️⬇️ 💽 Adjust here if you want to load the data from your google drive
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False
YOUR_NAME = "niclas"
SEPARATOR = ";" # Separator for the CSV file, e.g., ";" for semicolon, "," for comma
### ⬆️⬆️⬆️

### Load the data

In [ ]:
if IN_COLAB and USE_YOUR_DATA: # and confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

#### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
if USE_YOUR_DATA:
    GOLD_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.pp.label.{YOUR_NAME}.csv"
    PRED_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.pp.label.llm.csv"

    gold_df = pd.read_csv(GOLD_PATH, sep=SEPARATOR)
    pred_df = pd.read_csv(PRED_PATH, sep=SEPARATOR)

    print(f'{"Gold standard loaded:":<21} {len(gold_df)} entries')
    print(f'{"Predictions loaded:":<21} {len(pred_df)} entries')

    if len(gold_df) != len(pred_df):
        print("\n⚠️  Warning: Gold standard and predictions have different lengths!")
        print(f'{"":>4}{"Difference:":<12} {abs(len(gold_df) - len(pred_df))} entries')

    display(gold_df.head(3))
    display(pred_df.head(3))

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load the 'Armenpflege' Example

In [ ]:
if not USE_YOUR_DATA:
    GOLD_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.niclas.csv"
    PRED_URL = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.llm.csv"

    print('Loading gold standard ...', end='\r')
    gold_df = pd.read_csv(GOLD_URL, index_col="id", sep=SEPARATOR)

    print('Loading predictions   ...', end='\r')
    pred_df = pd.read_csv(PRED_URL, index_col="id")

    print(f'{"Gold standard loaded:":<21} {len(gold_df)} entries')
    print(f'{"Predictions loaded:":<21} {len(pred_df)} entries')

    if len(gold_df) != len(pred_df):
        print("\n⚠️  Warning: Gold standard and predictions have different lengths!")
        print(f'{"":>4}{"Difference:":<12} {abs(len(gold_df) - len(pred_df))} entries')

    print("Gold Data")
    display(gold_df.head(3))
    print("Predictions Data")
    display(pred_df.head(3))

## Merge the Dataframes

First, we merge the gold standard and prediction dataframes on the `id` column to obtain a unified dataframe containing both the gold label (`label`) and the predicted label (`predicted_label`). For this evaluation dataframe, we only keep the articles that have a gold label, discarding any unannotated entries.

In [ ]:
eval_df = (
    gold_df[gold_df['label'].notna()].merge(pred_df, on='id', how='left')
)

print(f'{"Gold standard entries:":<23} {len(gold_df[gold_df["label"].notna()])}')
print(f'{"Predictions entries:":<23} {len(pred_df)}')
print(f'{"Entries after merge:":<23} {len(eval_df)}')
display(eval_df.head(5))

## Inspect the Label Distribution

Before calculating any metrics, it is worth inspecting the label distribution. Are the classes roughly balanced, or are there large disparities in their frequency? Highly imbalanced label distributions are not only challenging for the model—rare classes can be difficult to learn—but also affect the meaningfulness of certain evaluation metrics, which may underrepresent and neglect rare classes, as we will see further below.

In [ ]:
# Inspect label distribution
label_freq = (
    eval_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)
label_freq["share"] = label_freq["count"] / label_freq["count"].sum()

pred_freq = (
    eval_df["predicted_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)
pred_freq["share"] = pred_freq["count"] / pred_freq["count"].sum()

In [ ]:
# Print statistics
print("Label Distribution in the Gold Standard")
display(label_freq.head())

In [ ]:
print("Label Distribution in the Predictions")
display(pred_freq.head())

Plotting the label distributions lets us take them in at a glance. By placing the gold and predicted distributions side by side, we gain a quick overview of how well the model has learned the true label distribution, and whether its predictions are reasonable or reveal any major issues.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, max(3, max(len(label_freq), len(pred_freq)) * 0.6)))

for ax, freq, title in zip(
    axes,
    [label_freq, pred_freq],
    ["Label Distribution in the Gold Standard", "Label Distribution in the Predictions"],
):
    bars = ax.barh(freq["label"], freq["count"], color="steelblue")
    for bar, share in zip(bars, freq["share"]):
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f"{share:.1%}",
            va="center",
            fontsize=9,
        )
    ax.set_xlabel("# Articles")
    ax.set_title(title)

plt.tight_layout()
plt.show()

## Confusion Matrix
A confusion matrix is a table used to evaluate the performance of a classification model. It compares the model’s predicted labels with the actual (true) labels, showing how many predictions were correct and where the model made mistakes. It provides detailed insight into the counts of true positives, false positives, false negatives, and true negatives, giving a more complete picture of the model’s performance.

For a binary classifier, the matrix has four cells:

|  | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

**How to Read It**
- **Diagonal cells** = correct predictions (should be high)
- **Off-diagonal cells** = errors (should be low)
- **Rows:** indicate the **recall** of a class (how many actual instances were correctly found)
- **Columns:** indicate the **precision** of a class (how many predicted instances were correct)

A confusion matrix gives a **detailed breakdown of model performance**, beyond just overall accuracy. It helps you:

- Identify specific types of errors (e.g., false positives vs. false negatives)
- Understand how well the model performs on each class
- Evaluate metrics like precision, recall, and F1-score
- Detect bias in the model (e.g., favoring one class over another)

This is especially useful in real-world tasks where different errors have different consequences (e.g., medical diagnosis or fraud detection).

In [ ]:
# Get true and predicted labels for evaluation
# Use predicted labels for the matrix to ensure that the error class is included
label_list = sorted(eval_df["predicted_label"].unique())
true_labels = eval_df["label"]
predicted_labels = eval_df["predicted_label"]

In [ ]:
confusion_matrix_df = pd.crosstab(
    true_labels,
    predicted_labels,
    rownames=["Gold Standard"],
    colnames=["Predicted"],
)

# Include the error class in the confusion matrix even if it has zero counts
confusion_matrix_df = confusion_matrix_df.reindex(index=label_list, fill_value=0)

plt.figure(figsize=(9, 7))

# Confusion matrix with absolute counts
sns.heatmap(confusion_matrix_df, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, square=True,
            cbar_kws={"shrink": 0.8, "pad": 0.02})

plt.title("Confusion Matrix", pad=20, fontsize=14, fontweight='bold')
plt.xlabel("Prediction", fontweight='bold', labelpad=15)
plt.ylabel("Gold Standard", fontweight='bold', labelpad=15)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout(pad=2.0)
plt.show()

The raw confusion matrix shows counts, which can be hard to compare when classes have very different numbers of instances. By normalizing the confusion matrix, we convert these counts into proportions, making it easier to interpret and compare performance across classes.

**Row-normalized (recall perspective):**  
When we normalize by rows, we obtain the recall perspective. We divide each value in a row by the total number of instances in that row (i.e., the total number of true instances of that class). For each true class (gold standard), we measure how many instances were correctly predicted (true positives) relative to the total number of actual instances in that class (true positives + false negatives). This lets us see, for each true class, how its instances are distributed across predictions. In this view, the diagonal values correspond to the recall for each class—the proportion of true instances that were correctly identified (true positives / (true positives + false negatives)).

**Column-normalized (precision perspective):**  
When we normalize by columns, we obtain the precision perspective. We divide each value in a column by the total number of predictions in that column (i.e., the total number of instances predicted as that class). For each predicted class, we measure how many instances truly belong to that class (true positives) compared to all instances predicted as that class (true positives + false positives). This lets us see, for each predicted class, how reliable those predictions are. In this view, the diagonal values correspond to the precision for each class—the proportion of predicted instances that actually belong to that class (true positives / (true positives + false positives)).

In [ ]:
# Normalized by row sum → Diagonal = Recall per class
cm_norm_recall = confusion_matrix_df.div(confusion_matrix_df.sum(axis=1), axis=0).fillna(0).round(2)
# Normalized by column sum → Diagonal = Precision per class
cm_norm_precision = confusion_matrix_df.div(confusion_matrix_df.sum(axis=0), axis=1).fillna(0).round(2)

# Remove error class: keep only gold labels in both rows and columns
gold_label_list = sorted(eval_df["label"].unique())
cm_norm_recall = cm_norm_recall.loc[
    cm_norm_recall.index.isin(gold_label_list),
    cm_norm_recall.columns.isin(gold_label_list)
]
cm_norm_precision = cm_norm_precision.loc[
    cm_norm_precision.index.isin(gold_label_list),
    cm_norm_precision.columns.isin(gold_label_list)
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, cm, title in zip(
    axes,
    [cm_norm_recall, cm_norm_precision],
    ["Normed Confusion Matrix\nRecall Perspective", "Normed Confusion Matrix\nPrecision Perspective"],
):
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                vmin=0, vmax=1, linewidths=0.5, square=True,
                cbar_kws={"shrink": 0.8, "pad": 0.02}, ax=ax)
    ax.set_title(title, pad=20, fontsize=14, fontweight='bold')
    ax.set_xlabel("Predicted", fontweight='bold', labelpad=15)
    ax.set_ylabel("Gold Standard", fontweight='bold', labelpad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout(pad=2.0)
plt.show()

## Metrics

To evaluate our classification model, we consider the following four metrics:

- **Recall:** Of all instances that truly belong to class X, how many did the model correctly identify?
- **Precision:** Of all instances predicted to belong to class X, how many actually belong to class X?
- **F1-score:** The harmonic mean of precision and recall, providing a balanced measure of both.
- **Accuracy:** The proportion of all instances that were correctly classified.

To calculate all relevant metrics, we can use scikit-learn's `classification_report()` function.


### Case-specific remarks
In our case, the predictions contain an additional class `error`, which does not appear in the gold standard. To evaluate the performance for each class, we count `error` predictions as false negatives of the corresponding gold standard class. However, we do not calculate any metrics for the `error` class itself. Consequently, the `error` class is also excluded from the aggregate metrics reported further below.

### Recall, Precision & F1-Score per Class

In [ ]:
gold_label_list = sorted(eval_df["label"].unique())

# Pass the gold label list to sklearn
report = classification_report(
    true_labels, predicted_labels,
    labels=gold_label_list,
    zero_division=0,
    output_dict=True,
)

print(report)
display_rows = gold_label_list
report_df = pd.DataFrame(report).T.loc[display_rows]
display(report_df.round(3).head(10))

### Aggregate Metrics across Classes

#### Micro Average vs. Macro Average

If we want to report average scores across classes, there are several ways to do so.


**Micro Average**

Micro-averaging pools the True Positives, False Positives and False Negatives across all classes together before computing the final metric. Hence, it gives equal importance to every single **instance**. It focuses on large classes. 

It is ideal when you want to measure overall performance, especially if class distributions are balanced. Because every instance counts equally, large, frequent classes will heavily influence the score.

It is stronlgy related to **accuracy**. In standard multi-class single-label classification, micro-averaged precision, recall, and F1-score all evaluate to the exact same value as overall model accuracy. Micro-averaging is typically used when multiple labels can be assigned to each instance or when evaluating a subset of classes. In our case, this applies because we exclude the `error` class and consider only the three gold standard classes when computing the micro-averaged scores.


**Macro Average**

Macro-averaging calculates the metrics for each class independently and then takes the unweighted mean. It gives equal importance to all **classes**, regardless of how many samples belong to it. It focuses on small classes.

It is highly recommended when your dataset is imbalanced and minority classes matter just as much as the majority ones. A poor score on a minority class will severely drag down the macro average, preventing the model from neglecting minority classes and from just "cheating" by always predicting the majority class.


**Weighted Average**

Weighted-averaging calculates the metrics for each class independently and then takes the weighted mean, i.e. it multiplies each class score by its support (=number of true instances in that class) before dividing by the total number of samples. It focuses on large classes.

It is forgiving of poor perormance in minority classes and focuses on larger classes.

Use weighted average when you want an overall performance metric that respects the natural, real-world distribution of your classes, but you still want the calculation to happen at the class level. It is ideal when underperforming on a rare class has zero or little negative business or operational impact.

In [ ]:
display_rows = gold_label_list + ["micro avg", "macro avg", "weighted avg"]
report_df = pd.DataFrame(report).T.loc[display_rows]
display(report_df.round(3).head(10))

#### Accuracy

Accuracy is the most intuitive metric, but it is also the most dangerous when misused. It simply calculates the percentage of total correct predictions out of all predictions made. Hence, it focuses on **global correctness**.

It is highly vulnerable to class imbalance, to the point of being completely misleading.

If your dataset is skewed, a model can cheat to achieve high accuracy. For example, if you are testing for a rare medical condition that only 1% of the population has, a broken model that predicts "Healthy" for every single patient will still achieve a staggering 99% accuracy, despite being completely useless.

Use accuracy when:
- Your classes are very **balanced**.
- True Positives and True Negatives carry the exact same weight, cost, and importance to you.
- You need a simple, universal metric to explain model performance to non-technical stakeholders.

Do **NOT** use accuracy when:
- Your dataset is imbalanced (even a 70/30 split can start to distort accuracy).
- The cost of a False Negative is vastly different from a False Positive. For example, in cancer screening, telling a sick patient they are healthy (False Negative) is catastrophic, while telling a healthy patient they need a second test (False Positive) is just inconvenient. Accuracy treats both errors as equal failures.

In [ ]:
accuracy = accuracy_score(true_labels, predicted_labels)
print(f'\nAccuracy : {accuracy:.3f}')

## Inter Annotator Agreement
When multiple annotators label the same data independently, they will not always agree. Inter-annotator agreement (IAA) measures how consistently different annotators assign labels to the same items. A high agreement suggests that the annotation task is well-defined and that the labels are reliable; low agreement may indicate ambiguity in the category definitions or the data itself.

A commonly used metric is Cohen's κ (kappa), which measures agreement while accounting for the level of agreement that would be expected by chance alone. A κ of 1 indicates perfect agreement, 0 indicates agreement no better than chance, and negative values indicate systematic disagreement.

Here, we compute Cohen's κ between two human annotators as a baseline, and then compare it against the agreement between a human annotator and the LLM.

In [ ]:
NICLAS_PATH = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.niclas.csv"
niclas_df = pd.read_csv(NICLAS_PATH, index_col="id", sep=SEPARATOR)
niclas_df = niclas_df[niclas_df["label"].notna()]

ELIAS_PATH = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.elias.csv"
elias_df = pd.read_csv(ELIAS_PATH, index_col="id", sep=SEPARATOR)
elias_df = elias_df[elias_df["label"].notna()]

display(niclas_df.head(3))
display(elias_df.head(3))

Merge the two annotation dataframes, keeping only entries for which both Niclas and Elias have provided a label. Then, calculate Cohen's Kappa.

In [ ]:
# rename label columns to avoid confusion after merge
niclas_df = niclas_df.rename(columns={"label": "niclas_label"})
elias_df = elias_df.rename(columns={"label": "elias_label"})

kappa_df = (
    niclas_df[niclas_df['niclas_label'].notna()].merge(elias_df, on='id', how='left')
)

kappa_human = cohen_kappa_score(kappa_df["niclas_label"], kappa_df["elias_label"])
print(f"Cohen's κ between Niclas and Elias: {kappa_human:.3f}")

Calculate Cohen's κ between Niclas and the LLM, then compare it against the human–human inter-annotator agreement to assess how closely the model's labels align with human judgement.

In [ ]:
kappa_llm = cohen_kappa_score(true_labels, predicted_labels)
print(f"Cohen's κ between Niclas and the LLM: {kappa_llm:.3f}")

## Overview
Choose your metric based on your data and goal:

- **Accuracy:** Use only if classes are perfectly **balanced**. Accuracy rewards models that over-predict frequent labels, making it highly misleading for imbalanced data.

- **Precision:** Use when you want to minimize false positives (costs of being wrong are high), e.g. when a high number of false alarms creates severe operational bottlenecks.  

- **Recall:** Use when you want to minimize false negatives (you cannot afford to miss a single positive case). Recall protects rare labels by measuring how many of them were actually found.  .

- **F1-Score:** Use when you need a balance between Precision and Recall, especially on imbalanced data. It is the harmonic mean of both.  

- **Macro Average:** Use when classes are imbalanced, but every class—particularly the small ones—is equally important.

- **Weighted Average:** Use when classes are imbalanced, and you want to reflect real-world proportions at a class level.

- **Micro Average:** Use when classes are imbalanced, and you care only about the total volume of correct individual instances. Use in multi-label scenarios. It is equivalent to accuracy in single-label scenarios.

- **Cohen's Kappa:** Use to evaluate classification performance by accounting for the possibility of agreements occurring by chance, especially on imbalanced data or multi-class problems.

In [ ]:
summary = pd.DataFrame({
    "metric": ["Macro Precision", "Macro Recall", "Macro F1", "Accuracy", "Cohen's κ (LLM)", "Cohen's κ (Human)"],
    'value': [
        report['macro avg']['precision'],
        report['macro avg']['recall'],
        report['macro avg']['f1-score'],
        accuracy,
        kappa_llm,
        kappa_human
    ]
})

display(summary.style.hide(axis='index').format({"value": "{:.3f}"}))

## Inspect Specific Misclassification Types
To gain a deeper understanding of the model's errors, you can inspect individual examples for any misclassification type of your choice. Based on the confusion matrix above, the most common error is the model predicting "Meinungsartikel" for entries that actually belong to "Unspezifischer Bericht". Examining these cases can reveal patterns that explain the confusion and help you improve the prompt.

In [ ]:
# ⬇️⬇️⬇️ Adjust the label combination you want to inspect
TRUE_LABEL = "Unspezifischer Bericht"
PRED_LABEL = "Meinungsartikel"
# ⬆️⬆️⬆️

In [ ]:
misclassified = eval_df[
(eval_df["label"] == TRUE_LABEL) &
(eval_df["predicted_label"] == PRED_LABEL)
]

print(f"Found {len(misclassified)} misclassified article(s).\n")
print("=" * 150)

for id, row in misclassified.iterrows():
    print(f"ID: {id}")
    print(f"Human Label: {row['label']}")
    print(f"LLM Predicted Label: {row['predicted_label']}", end="\n\n")
    if "reasoning" in row:
        print("Reasoning:")
        print(textwrap.fill(str(row['reasoning']), width=150), end="\n\n")
    if "pseudo_paragraph" in row:
        print("Article Pseudo-Paragraph:")
        print(textwrap.fill(str(row['pseudo_paragraph']), width=150))
    print("=" * 150)